# TIA Risk Prediction: Effect of Influenza Vaccines on First Occurrence of Transient Ischemic Attack

**Research Question:** What are the major predictors of first occurrence of TIA, with specific focus on the effect of influenza vaccination?

**Data Source:** Pooled NHANES (National Health and Nutrition Examination Survey) — public CDC longitudinal health surveys containing:
- Stroke/TIA outcomes (Medical Conditions questionnaire)
- Influenza vaccination history (Immunization questionnaire)
- Demographics, comorbidities, lab values, lifestyle factors

**Models:** Logistic Regression, Decision Tree, Random Forest, Gradient Boosting, XGBoost, LightGBM, PyTorch MLP (Neural Network)

**Cycles Pooled:** 2015–2016, 2017–2018, 2017–2020 (pre-pandemic)

---

## 0. Environment Setup
Run once to install dependencies.

In [1]:
import subprocess, sys

packages = [
    'pandas>=1.5', 'numpy>=1.23', 'scikit-learn>=1.2',
    'xgboost>=1.7', 'lightgbm>=3.3', 'optuna>=3.0',
    'shap>=0.41', 'matplotlib>=3.6', 'seaborn>=0.12',
    'torch>=2.0', 'imbalanced-learn>=0.10',
    'pyreadstat>=1.2', 'requests>=2.28', 'tqdm>=4.64', 'scipy>=1.9'
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + packages, check=True)
print('All packages installed successfully.')

All packages installed successfully.


## 1. Imports & Global Configuration

In [2]:
import os
import warnings
import pickle
import requests
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    classification_report, confusion_matrix, roc_curve,
    precision_recall_curve, ConfusionMatrixDisplay, recall_score,
    precision_score, accuracy_score
)

import xgboost as xgb
import lightgbm as lgb
import optuna
from imblearn.over_sampling import SMOTE
import shap

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)
torch.manual_seed(42)

DATA_DIR = Path('data/nhanes')
DATA_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print('Imports complete.')

c:\Users\tukum\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu
Imports complete.


## 2. Data Acquisition — Pooled NHANES

We pool three NHANES cycles (2015-2016, 2017-2018, 2017-2020 pre-pandemic) to maximize sample size.
Each cycle includes: Demographics, Medical Conditions (stroke/TIA), Immunization (flu vaccine),
Blood Pressure, Diabetes, Smoking, BMI, Cholesterol, Alcohol, Physical Activity.

In [3]:
NHANES_BASE = 'https://wwwn.cdc.gov/nchs/nhanes'

NHANES_FILES = {
    # ── 2015-2016 (cycle I) ─────────────────────────────────────────────────
    'DEMO_I':  f'{NHANES_BASE}/2015-2016/DEMO_I.XPT',
    'MCQ_I':   f'{NHANES_BASE}/2015-2016/MCQ_I.XPT',
    'IMQ_I':   f'{NHANES_BASE}/2015-2016/IMQ_I.XPT',
    'BPQ_I':   f'{NHANES_BASE}/2015-2016/BPQ_I.XPT',
    'DIQ_I':   f'{NHANES_BASE}/2015-2016/DIQ_I.XPT',
    'SMQ_I':   f'{NHANES_BASE}/2015-2016/SMQ_I.XPT',
    'BMX_I':   f'{NHANES_BASE}/2015-2016/BMX_I.XPT',
    'BPX_I':   f'{NHANES_BASE}/2015-2016/BPX_I.XPT',
    'TCHOL_I': f'{NHANES_BASE}/2015-2016/TCHOL_I.XPT',
    'HDL_I':   f'{NHANES_BASE}/2015-2016/HDL_I.XPT',
    'ALQ_I':   f'{NHANES_BASE}/2015-2016/ALQ_I.XPT',
    'PAQ_I':   f'{NHANES_BASE}/2015-2016/PAQ_I.XPT',
    # ── 2017-2018 (cycle J) ─────────────────────────────────────────────────
    'DEMO_J':  f'{NHANES_BASE}/2017-2018/DEMO_J.XPT',
    'MCQ_J':   f'{NHANES_BASE}/2017-2018/MCQ_J.XPT',
    'IMQ_J':   f'{NHANES_BASE}/2017-2018/IMQ_J.XPT',
    'BPQ_J':   f'{NHANES_BASE}/2017-2018/BPQ_J.XPT',
    'DIQ_J':   f'{NHANES_BASE}/2017-2018/DIQ_J.XPT',
    'SMQ_J':   f'{NHANES_BASE}/2017-2018/SMQ_J.XPT',
    'BMX_J':   f'{NHANES_BASE}/2017-2018/BMX_J.XPT',
    'BPX_J':   f'{NHANES_BASE}/2017-2018/BPX_J.XPT',
    'TCHOL_J': f'{NHANES_BASE}/2017-2018/TCHOL_J.XPT',
    'HDL_J':   f'{NHANES_BASE}/2017-2018/HDL_J.XPT',
    'ALQ_J':   f'{NHANES_BASE}/2017-2018/ALQ_J.XPT',
    'PAQ_J':   f'{NHANES_BASE}/2017-2018/PAQ_J.XPT',
    # ── 2017-2020 pre-pandemic (cycle P) ────────────────────────────────────
    'DEMO_P':  f'{NHANES_BASE}/2017-2020/DEMO_P.XPT',
    'MCQ_P':   f'{NHANES_BASE}/2017-2020/MCQ_P.XPT',
    'IMQ_P':   f'{NHANES_BASE}/2017-2020/IMQ_P.XPT',
    'BPQ_P':   f'{NHANES_BASE}/2017-2020/BPQ_P.XPT',
    'DIQ_P':   f'{NHANES_BASE}/2017-2020/DIQ_P.XPT',
    'SMQ_P':   f'{NHANES_BASE}/2017-2020/SMQ_P.XPT',
    'BMX_P':   f'{NHANES_BASE}/2017-2020/BMX_P.XPT',
    'BPXO_P':  f'{NHANES_BASE}/2017-2020/BPXO_P.XPT',
    'TCHOL_P': f'{NHANES_BASE}/2017-2020/TCHOL_P.XPT',
    'HDL_P':   f'{NHANES_BASE}/2017-2020/HDL_P.XPT',
    'ALQ_P':   f'{NHANES_BASE}/2017-2020/ALQ_P.XPT',
    'PAQ_P':   f'{NHANES_BASE}/2017-2020/PAQ_P.XPT',
}


def _is_valid_xpt(path):
    """SAS XPORT v5 files always start with 'HEADER RECORD*'."""
    try:
        with open(path, 'rb') as f:
            return f.read(14) == b'HEADER RECORD*'
    except Exception:
        return False


def download_xpt(name, url):
    fpath = DATA_DIR / f'{name}.XPT'

    # Delete cached file if it's corrupt (e.g. an HTML error page).
    if fpath.exists() and not _is_valid_xpt(fpath):
        print(f'  [CORRUPT] {name}: cached file is not a valid XPT — deleting and re-downloading')
        fpath.unlink()

    if not fpath.exists():
        try:
            headers = {'User-Agent': 'Mozilla/5.0 (compatible; NHANES-downloader/1.0)'}
            r = requests.get(url, timeout=180, headers=headers)
            r.raise_for_status()
            if not r.content.startswith(b'HEADER RECORD'):
                print(f'  [SKIP] {name}: server returned non-XPT content '
                      f'(starts with {r.content[:60]!r})')
                return None
            fpath.write_bytes(r.content)
        except Exception as e:
            print(f'  [SKIP] {name}: {e}')
            return None

    try:
        return pd.read_sas(fpath, format='xport', encoding='iso-8859-1')
    except Exception as e:
        print(f'  [READ-ERR] {name}: {e}')
        return None


print('Downloading NHANES data (cached after first run)...')
nhanes = {}
for name, url in NHANES_FILES.items():
    df_ = download_xpt(name, url)
    if df_ is not None:
        nhanes[name] = df_
        print(f'  {name:<10} {len(df_):>6} rows  {len(df_.columns):>3} cols')

print(f'\nTotal modules loaded: {len(nhanes)}')

  DEMO_I       9971 rows   47 cols
  MCQ_I        9575 rows   90 cols
  IMQ_I        9971 rows    8 cols
  BPQ_I        6327 rows   11 cols
  DIQ_I        9575 rows   54 cols
  SMQ_I        7001 rows   42 cols
  BMX_I        9544 rows   26 cols
  BPX_I        9544 rows   21 cols
  TCHOL_I      8021 rows    3 cols
  HDL_I        8021 rows    3 cols
  ALQ_I        5735 rows   10 cols
  PAQ_I        9255 rows   94 cols
  DEMO_J       9254 rows   46 cols
  MCQ_J        8897 rows   76 cols
  IMQ_J        9254 rows   11 cols
  BPQ_J        6161 rows   11 cols
  DIQ_J        8897 rows   54 cols
  SMQ_J        6724 rows   37 cols
  BMX_J        8704 rows   21 cols
  BPX_J        8704 rows   21 cols
  TCHOL_J      7435 rows    3 cols
  HDL_J        7435 rows    3 cols
  ALQ_J        5533 rows   10 cols
  PAQ_J        5856 rows   17 cols
  DEMO_P      15560 rows   29 cols
  MCQ_P       14986 rows   63 cols
  IMQ_P       15560 rows   11 cols
  BPQ_P       10195 rows   11 cols
  DIQ_P       14986 

## 3. Data Integration — Merge Modules per Cycle, then Pool

In [4]:
def safe_cols(df, wanted):
    """Return only columns that exist in df."""
    return [c for c in wanted if c in df.columns]


def merge_cycle(suffix):
    """Merge all NHANES modules for a single survey cycle on SEQN."""
    demo_key = f'DEMO_{suffix}'
    if demo_key not in nhanes:
        return None

    # ── demographics ──────────────────────────────────────────────────────
    demo_cols = safe_cols(nhanes[demo_key], [
        'SEQN', 'RIAGENDR', 'RIDAGEYR', 'RIDRETH1', 'DMDEDUC2', 'INDFMPIR', 'DMDMARTZ'
    ])
    merged = nhanes[demo_key][demo_cols].copy()

    def left_merge(mod_key, cols):
        if mod_key not in nhanes:
            return
        keep = safe_cols(nhanes[mod_key], ['SEQN'] + cols)
        nonlocal merged
        merged = merged.merge(nhanes[mod_key][keep], on='SEQN', how='left')

    # ── medical conditions (stroke = MCQ160F) ─────────────────────────────
    left_merge(f'MCQ_{suffix}', ['MCQ160F', 'MCQ160E', 'MCQ160B', 'MCQ160D', 'MCQ220'])

    # ── immunization (flu vaccine) ─────────────────────────────────────────
    left_merge(f'IMQ_{suffix}', ['IMQ011', 'IMQ020', 'IMQ011A'])

    # ── blood pressure questionnaire ───────────────────────────────────────
    left_merge(f'BPQ_{suffix}', ['BPQ020', 'BPQ040A', 'BPQ050A'])

    # ── diabetes ──────────────────────────────────────────────────────────
    left_merge(f'DIQ_{suffix}', ['DIQ010', 'DIQ050', 'DIQ070'])

    # ── smoking ───────────────────────────────────────────────────────────
    left_merge(f'SMQ_{suffix}', ['SMQ020', 'SMQ040'])

    # ── body measures ─────────────────────────────────────────────────────
    left_merge(f'BMX_{suffix}', ['BMXBMI', 'BMXWAIST'])

    # ── blood pressure exam (column names differ by cycle) ─────────────────
    bp_key = f'BPXO_{suffix}' if f'BPXO_{suffix}' in nhanes else f'BPX_{suffix}'
    left_merge(bp_key, ['BPXSY1', 'BPXDI1', 'BPXOSY1', 'BPXODI1'])

    # ── cholesterol ───────────────────────────────────────────────────────
    left_merge(f'TCHOL_{suffix}', ['LBXTC'])
    left_merge(f'HDL_{suffix}',   ['LBDHDD', 'LBXHDD'])

    # ── alcohol ───────────────────────────────────────────────────────────
    left_merge(f'ALQ_{suffix}', ['ALQ121', 'ALQ130', 'ALQ111'])

    # ── physical activity ─────────────────────────────────────────────────
    left_merge(f'PAQ_{suffix}', ['PAQ605', 'PAQ620', 'PAQ650', 'PAD680'])

    merged['cycle'] = suffix
    return merged


cycle_frames = []
for s in ['I', 'J', 'P']:
    f = merge_cycle(s)
    if f is not None:
        cycle_frames.append(f)
        print(f'Cycle {s}: {len(f):,} participants, {f.shape[1]} columns')

if not cycle_frames:
    missing_demos = [f'DEMO_{s}' for s in ['I', 'J', 'P'] if f'DEMO_{s}' not in nhanes]
    raise RuntimeError(
        f'No NHANES cycles loaded — cycle_frames is empty.\n'
        f'nhanes dict has {len(nhanes)} modules loaded.\n'
        f'Missing DEMO files: {missing_demos}\n'
        f'Re-run the download cell above to fetch (or re-fetch) the data.'
    )

df_raw = pd.concat(cycle_frames, ignore_index=True)
print(f'\nPooled dataset: {len(df_raw):,} rows  x  {df_raw.shape[1]} columns')
print(df_raw.dtypes.value_counts())

Cycle I: 9,971 participants, 33 columns
Cycle J: 9,254 participants, 35 columns
Cycle P: 15,560 participants, 36 columns

Pooled dataset: 34,785 rows  x  38 columns
float64    37
str         1
Name: count, dtype: int64


## 4. Target Variable & Feature Engineering

In [ ]:
df = df_raw.copy()

# ── Target: stroke/TIA (MCQ160F: 1=Yes, 2=No, 7/9=refused/unknown) ────────
df['stroke_tia'] = np.where(df['MCQ160F'] == 1, 1,
                    np.where(df['MCQ160F'] == 2, 0, np.nan))
df = df.dropna(subset=['stroke_tia']).copy()
df['stroke_tia'] = df['stroke_tia'].astype(int)

# ── Flu vaccine ────────────────────────────────────────────────────────────
# IMQ011: 1=Yes, 2=No (flu shot past 12 months) — harmonise across cycles
if 'IMQ011' in df.columns and 'IMQ011A' in df.columns:
    flu_raw = df['IMQ011'].fillna(df['IMQ011A'])
elif 'IMQ011' in df.columns:
    flu_raw = df['IMQ011']
elif 'IMQ011A' in df.columns:
    flu_raw = df['IMQ011A']
else:
    flu_raw = pd.Series(np.nan, index=df.index)

df['flu_vaccine'] = np.where(flu_raw == 1, 1, np.where(flu_raw == 2, 0, np.nan))

# ── Age ─────────────────────────────────────────────────────────────────────
df['age']      = df['RIDAGEYR']
df['age_sq']   = df['RIDAGEYR'] ** 2
df['is_senior'] = (df['RIDAGEYR'] >= 65).astype(int)

# Key hypothesis: vaccine x senior interaction
df['flu_x_senior'] = df['flu_vaccine'].fillna(0) * df['is_senior']

# ── Sex ─────────────────────────────────────────────────────────────────────
df['male'] = (df['RIAGENDR'] == 1).astype(int)

# ── Hypertension ────────────────────────────────────────────────────────────
df['hypertension'] = np.where(df['BPQ020'] == 1, 1,
                     np.where(df['BPQ020'] == 2, 0, np.nan))
# Augment with measured BP (SBP >= 140 or DBP >= 90)
for sys_col in ['BPXOSY1', 'BPXSY1']:
    if sys_col in df.columns:
        df['sbp'] = df.get('sbp', pd.Series(np.nan, index=df.index)).fillna(df[sys_col])
for dia_col in ['BPXODI1', 'BPXDI1']:
    if dia_col in df.columns:
        df['dbp'] = df.get('dbp', pd.Series(np.nan, index=df.index)).fillna(df[dia_col])

# NEW — paste these 7 lines
if 'sbp' in df.columns:
    _dbp = df.get('dbp', pd.Series(90, index=df.index))
    _high_bp = (df['sbp'] >= 140) | (_dbp >= 90)
    _missing = df['hypertension'].isna()
    df.loc[_missing & _high_bp,  'hypertension'] = 1
    df.loc[_missing & ~_high_bp, 'hypertension'] = 0
    df['pulse_pressure'] = df['sbp'] - df.get('dbp', pd.Series(np.nan, index=df.index))

# ── Diabetes ────────────────────────────────────────────────────────────────
df['diabetes'] = np.where(df['DIQ010'] == 1, 1,
                 np.where(df['DIQ010'] == 2, 0, np.nan))

# ── Smoking ─────────────────────────────────────────────────────────────────
df['ever_smoked']    = np.where(df['SMQ020'] == 1, 1, np.where(df['SMQ020'] == 2, 0, np.nan))
df['current_smoker'] = np.where(df.get('SMQ040', pd.Series(np.nan, index=df.index)) == 1, 1,
                       np.where(df.get('SMQ040', pd.Series(np.nan, index=df.index)) == 2, 0, np.nan))

# ── BMI ─────────────────────────────────────────────────────────────────────
df['bmi']   = df['BMXBMI']
df['obese'] = np.where(df['BMXBMI'] >= 30, 1, np.where(df['BMXBMI'].notna(), 0, np.nan))

# ── Cholesterol ──────────────────────────────────────────────────────────────
df['total_chol'] = df['LBXTC']
hdl_col = 'LBDHDD' if 'LBDHDD' in df.columns else 'LBXHDD'
if hdl_col in df.columns:
    df['hdl']     = df[hdl_col]
    df['low_hdl'] = np.where(df[hdl_col] < 40, 1, np.where(df[hdl_col].notna(), 0, np.nan))

# ── Cardiac comorbidities ────────────────────────────────────────────────────
for col, name in [('MCQ160E', 'chd'), ('MCQ160B', 'chf'), ('MCQ160D', 'angina')]:
    if col in df.columns:
        df[name] = np.where(df[col] == 1, 1, np.where(df[col] == 2, 0, np.nan))
cardiac_cols = [c for c in ['chd', 'chf', 'angina'] if c in df.columns]
if cardiac_cols:
    df['cardiac_score'] = df[cardiac_cols].fillna(0).sum(axis=1)

# ── Cancer history ───────────────────────────────────────────────────────────
if 'MCQ220' in df.columns:
    df['cancer_hx'] = np.where(df['MCQ220'] == 1, 1, np.where(df['MCQ220'] == 2, 0, np.nan))

# ── Socioeconomic ────────────────────────────────────────────────────────────
df['in_poverty']    = np.where(df['INDFMPIR'] < 1.0, 1, np.where(df['INDFMPIR'].notna(), 0, np.nan))
df['low_education'] = np.where(df['DMDEDUC2'].isin([1, 2]), 1,
                      np.where(df['DMDEDUC2'].isin([3, 4, 5]), 0, np.nan))

# ── Alcohol ──────────────────────────────────────────────────────────────────
alc_col = next((c for c in ['ALQ130', 'ALQ121'] if c in df.columns), None)
if alc_col:
    df['heavy_alcohol'] = np.where(df[alc_col] >= 14, 1, np.where(df[alc_col].notna(), 0, np.nan))

# ── Sedentary behaviour ──────────────────────────────────────────────────────
if 'PAD680' in df.columns:
    df['sedentary'] = np.where(df['PAD680'] >= 600, 1, np.where(df['PAD680'].notna(), 0, np.nan))

# ── Race/ethnicity dummies ────────────────────────────────────────────────────
race_map = {1:'race_mex_amer', 2:'race_other_hisp', 3:'race_white', 4:'race_black', 6:'race_asian'}
for code, col in race_map.items():
    df[col] = (df['RIDRETH1'] == code).astype(int)

print(f'Dataset after target filtering: {len(df):,} rows')
print(f"\nTarget distribution:")
vc = df['stroke_tia'].value_counts()
print(f"  No Stroke/TIA : {vc.get(0,0):,} ({vc.get(0,0)/len(df)*100:.1f}%)")
print(f"  Stroke/TIA    : {vc.get(1,0):,} ({vc.get(1,0)/len(df)*100:.1f}%)")
print(f"\nFlu vaccine coverage: {df['flu_vaccine'].mean()*100:.1f}% vaccinated")

TypeError: "value" parameter must be a scalar, dict or Series, but you passed a "ndarray"

## 5. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# Age distribution by stroke status
ax = axes[0, 0]
for label, grp in df.groupby('stroke_tia'):
    grp['age'].dropna().hist(ax=ax, bins=30, alpha=0.6,
                              label='Stroke/TIA' if label else 'No Stroke/TIA', density=True)
ax.set(title='Age Distribution by Stroke Status', xlabel='Age', ylabel='Density')
ax.legend()

# Flu vaccine vs stroke
ax = axes[0, 1]
ct = df.groupby('flu_vaccine')['stroke_tia'].mean().rename({0:'No Vaccine', 1:'Vaccinated'})
ct.plot(kind='bar', ax=ax, color=['#e74c3c', '#2ecc71'], rot=0, edgecolor='black')
ax.set(title='Stroke/TIA Rate by Flu Vaccine Status', ylabel='Proportion with Stroke/TIA')

# Stroke rate by age group
ax = axes[0, 2]
df['age_grp'] = pd.cut(df['age'], bins=[0, 44, 54, 64, 74, 120],
                        labels=['<45', '45-54', '55-64', '65-74', '75+'])
df.groupby('age_grp')['stroke_tia'].mean().plot(kind='bar', ax=ax, rot=0, color='steelblue', edgecolor='black')
ax.set(title='Stroke/TIA Rate by Age Group', ylabel='Proportion')

# Comorbidity correlation with stroke
ax = axes[1, 0]
risk_cols = ['hypertension', 'diabetes', 'obese', 'ever_smoked', 'flu_vaccine']
risk_cols = [c for c in risk_cols if c in df.columns]
corrs = df[risk_cols + ['stroke_tia']].corr()['stroke_tia'].drop('stroke_tia').sort_values()
corrs.plot(kind='barh', ax=ax, color=['#e74c3c' if v > 0 else '#2ecc71' for v in corrs])
ax.axvline(0, color='black', linewidth=0.8)
ax.set(title='Correlation with Stroke/TIA', xlabel='Pearson r')

# BMI distribution
ax = axes[1, 1]
for label, grp in df.groupby('stroke_tia'):
    grp['bmi'].dropna().hist(ax=ax, bins=30, alpha=0.6,
                              label='Stroke/TIA' if label else 'No Stroke/TIA', density=True)
ax.set(title='BMI Distribution by Stroke Status', xlabel='BMI')
ax.legend()

# Flu vaccine x senior interaction
ax = axes[1, 2]
tbl = df.groupby(['is_senior', 'flu_vaccine'])['stroke_tia'].mean().unstack()
tbl.index = ['< 65 yrs', '≥ 65 yrs']
tbl.columns = ['No Vaccine', 'Vaccinated']
tbl.plot(kind='bar', ax=ax, rot=0, edgecolor='black')
ax.set(title='Vaccine Effect on Stroke by Age Group', ylabel='Proportion Stroke/TIA')

plt.suptitle('NHANES Pooled — Exploratory Data Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_overview.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Missing value heatmap & descriptive statistics
eng_cols = [
    'age', 'male', 'flu_vaccine', 'is_senior', 'flu_x_senior',
    'hypertension', 'diabetes', 'ever_smoked', 'current_smoker',
    'bmi', 'obese', 'total_chol', 'cardiac_score', 'in_poverty',
    'low_education', 'stroke_tia'
]
eng_cols = [c for c in eng_cols if c in df.columns]

miss = df[eng_cols].isna().mean().sort_values(ascending=False)
print('Missing rate per engineered feature:')
print(miss[miss > 0].to_string())

print('\n--- Descriptive Statistics (Stroke vs No-Stroke) ---')
num_cols = [c for c in ['age', 'bmi', 'total_chol', 'sbp'] if c in df.columns]
print(df.groupby('stroke_tia')[num_cols].mean().T.rename(columns={0:'No Stroke', 1:'Stroke/TIA'}))

# Chi-square tests for categorical predictors
print('\n--- Chi-Square Tests (categorical vs stroke) ---')
cat_feats = ['hypertension', 'diabetes', 'flu_vaccine', 'ever_smoked', 'male', 'is_senior']
cat_feats = [c for c in cat_feats if c in df.columns]
for feat in cat_feats:
    ct = pd.crosstab(df[feat].dropna(), df.loc[df[feat].notna(), 'stroke_tia'])
    chi2, p, _, _ = stats.chi2_contingency(ct)
    print(f'  {feat:<20}  chi2={chi2:8.2f}  p={p:.4f}')

## 6. Feature Selection & Preprocessing

In [ ]:
CANDIDATE_FEATURES = [
    'age', 'age_sq', 'is_senior', 'male',
    'flu_vaccine', 'flu_x_senior',
    'hypertension', 'diabetes',
    'ever_smoked', 'current_smoker',
    'bmi', 'obese',
    'total_chol',
    'sbp', 'dbp', 'pulse_pressure',
    'cardiac_score',
    'cancer_hx',
    'in_poverty', 'low_education',
    'heavy_alcohol',
    'sedentary',
    'low_hdl', 'hdl',
    'race_white', 'race_black', 'race_asian', 'race_mex_amer', 'race_other_hisp',
]

FEATURE_COLS = [c for c in CANDIDATE_FEATURES if c in df.columns]
TARGET = 'stroke_tia'

df_model = df[FEATURE_COLS + [TARGET]].copy()

# Drop rows with fewer than 5 non-null features (near-empty records)
df_model = df_model[df_model[FEATURE_COLS].notna().sum(axis=1) >= 5]

print(f'Model dataset: {len(df_model):,} rows  x  {len(FEATURE_COLS)} features')
print(f'Features used: {FEATURE_COLS}')

In [ ]:
X = df_model[FEATURE_COLS].copy()
y = df_model[TARGET].copy()

# ── Stratified 70 / 15 / 15 split ─────────────────────────────────────────
X_tmp, X_test, y_tmp, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=0.1765, stratify=y_tmp, random_state=42
)

print(f'Train : {len(X_train):,}  |  Val : {len(X_val):,}  |  Test : {len(X_test):,}')
print(f'Class balance (train) — 0:{(y_train==0).mean()*100:.1f}%  1:{(y_train==1).mean()*100:.1f}%')

# ── Imputation (median) + scaling ─────────────────────────────────────────
imputer = SimpleImputer(strategy='median')
scaler  = StandardScaler()

X_train_imp = imputer.fit_transform(X_train)
X_val_imp   = imputer.transform(X_val)
X_test_imp  = imputer.transform(X_test)

X_train_sc  = scaler.fit_transform(X_train_imp)
X_val_sc    = scaler.transform(X_val_imp)
X_test_sc   = scaler.transform(X_test_imp)

# ── SMOTE on training set only ─────────────────────────────────────────────
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_sm, y_train_sm = smote.fit_resample(X_train_sc, y_train)
print(f'After SMOTE: {len(X_train_sm):,} training samples (balanced)')

## 7. Model Training

All models are evaluated on the same held-out validation set. Final ranking uses **ROC-AUC** as primary metric, **PR-AUC** as secondary (appropriate for class-imbalanced medical prediction).

In [ ]:
results = {}


def fit_and_score(name, model, X_tr, y_tr, X_v, y_v, use_scaled=True):
    """Fit model, evaluate on validation set, store results."""
    model.fit(X_tr, y_tr)
    y_prob = model.predict_proba(X_v)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)

    roc = roc_auc_score(y_v, y_prob)
    pr  = average_precision_score(y_v, y_prob)
    f1  = f1_score(y_v, y_pred, zero_division=0)
    sen = recall_score(y_v, y_pred, zero_division=0)
    spec = recall_score(y_v, y_pred, pos_label=0, zero_division=0)

    results[name] = dict(model=model, roc_auc=roc, pr_auc=pr, f1=f1,
                         sensitivity=sen, specificity=spec,
                         y_prob=y_prob, y_pred=y_pred,
                         uses_scaled=use_scaled)
    print(f'{name:<32}  ROC={roc:.4f}  PR={pr:.4f}  F1={f1:.4f}  Sen={sen:.4f}  Spec={spec:.4f}')
    return model


scale_pos_weight = float((y_train == 0).sum()) / float((y_train == 1).sum())
print(f'Class imbalance ratio (neg/pos): {scale_pos_weight:.2f}')
print(f'{"Model":<32}  {"ROC":>6}  {"PR":>6}  {"F1":>6}  {"Sens":>6}  {"Spec":>6}')
print('-' * 78)

In [ ]:
# ── Logistic Regression (baseline) ───────────────────────────────────────
lr = LogisticRegression(C=1.0, max_iter=2000, solver='lbfgs', random_state=42)
fit_and_score('Logistic Regression', lr, X_train_sm, y_train_sm, X_val_sc, y_val)

In [ ]:
# ── Decision Tree ─────────────────────────────────────────────────────────
dt = DecisionTreeClassifier(max_depth=8, min_samples_leaf=30, class_weight='balanced', random_state=42)
fit_and_score('Decision Tree', dt, X_train_sm, y_train_sm, X_val_sc, y_val)

In [ ]:
# ── Random Forest ─────────────────────────────────────────────────────────
rf = RandomForestClassifier(
    n_estimators=400, max_depth=12, min_samples_leaf=10,
    class_weight='balanced', n_jobs=-1, random_state=42
)
fit_and_score('Random Forest', rf, X_train_sm, y_train_sm, X_val_sc, y_val)

In [ ]:
# ── Gradient Boosting (sklearn) ───────────────────────────────────────────
gb = GradientBoostingClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=4,
    subsample=0.8, min_samples_leaf=20, random_state=42
)
fit_and_score('Gradient Boosting', gb, X_train_sm, y_train_sm, X_val_sc, y_val)

In [ ]:
# ── XGBoost ───────────────────────────────────────────────────────────────
xgb_model = xgb.XGBClassifier(
    n_estimators=500, learning_rate=0.05, max_depth=5,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss', random_state=42, n_jobs=-1, verbosity=0
)
fit_and_score('XGBoost', xgb_model, X_train_imp, y_train, X_val_imp, y_val, use_scaled=False)

In [ ]:
# ── LightGBM ──────────────────────────────────────────────────────────────
lgb_model = lgb.LGBMClassifier(
    n_estimators=500, learning_rate=0.05, max_depth=7,
    num_leaves=63, subsample=0.8, colsample_bytree=0.8,
    min_child_samples=20, class_weight='balanced',
    random_state=42, n_jobs=-1, verbose=-1
)
fit_and_score('LightGBM', lgb_model, X_train_imp, y_train, X_val_imp, y_val, use_scaled=False)

## 8. Deep Learning — PyTorch MLP

In [ ]:
class StrokeNet(nn.Module):
    """Feed-forward MLP with batch-norm, dropout, and residual skip connections."""

    def __init__(self, input_dim, hidden=(256, 128, 64), dropout=0.35):
        super().__init__()
        self.blocks = nn.ModuleList()
        in_d = input_dim
        for h in hidden:
            self.blocks.append(nn.Sequential(
                nn.Linear(in_d, h),
                nn.BatchNorm1d(h),
                nn.GELU(),
                nn.Dropout(dropout)
            ))
            in_d = h
        self.head = nn.Linear(in_d, 1)

    def forward(self, x):
        for block in self.blocks:
            x = block(x)
        return self.head(x).squeeze(1)


def train_stroke_net(X_tr, y_tr, X_v, y_v,
                     hidden=(256, 128, 64), dropout=0.35,
                     lr=3e-4, epochs=80, batch_size=256):
    pos_w = torch.tensor([(y_tr == 0).sum() / max((y_tr == 1).sum(), 1)],
                          dtype=torch.float32).to(DEVICE)

    ds   = TensorDataset(torch.FloatTensor(X_tr),
                          torch.FloatTensor(y_tr.values if hasattr(y_tr, 'values') else y_tr))
    dl   = DataLoader(ds, batch_size=batch_size, shuffle=True, drop_last=False)
    Xv_t = torch.FloatTensor(X_v).to(DEVICE)

    model     = StrokeNet(X_tr.shape[1], hidden, dropout).to(DEVICE)
    opt       = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, epochs=epochs,
                                               steps_per_epoch=len(dl))
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)

    best_auc, best_state = 0.0, None

    for epoch in range(epochs):
        model.train()
        for Xb, yb in dl:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = criterion(model(Xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            scheduler.step()

        model.eval()
        with torch.no_grad():
            logits = model(Xv_t).cpu()
        prob = torch.sigmoid(logits).numpy()
        auc  = roc_auc_score(y_v, prob)

        if auc > best_auc:
            best_auc   = auc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

        if (epoch + 1) % 20 == 0:
            print(f'  epoch {epoch+1:3d}  val_roc={auc:.4f}  best={best_auc:.4f}')

    model.load_state_dict(best_state)
    return model, best_auc


print('Training Neural Network (MLP)...')
nn_model, nn_best_auc = train_stroke_net(X_train_sc, y_train, X_val_sc, y_val)

# Evaluate on validation
nn_model.eval()
with torch.no_grad():
    logits_v = nn_model(torch.FloatTensor(X_val_sc).to(DEVICE)).cpu()
y_prob_nn = torch.sigmoid(logits_v).numpy()
y_pred_nn = (y_prob_nn >= 0.5).astype(int)

roc_nn = roc_auc_score(y_val, y_prob_nn)
pr_nn  = average_precision_score(y_val, y_prob_nn)
f1_nn  = f1_score(y_val, y_pred_nn, zero_division=0)
sen_nn = recall_score(y_val, y_pred_nn, zero_division=0)
spe_nn = recall_score(y_val, y_pred_nn, pos_label=0, zero_division=0)

results['Neural Network (MLP)'] = dict(
    model=nn_model, roc_auc=roc_nn, pr_auc=pr_nn, f1=f1_nn,
    sensitivity=sen_nn, specificity=spe_nn,
    y_prob=y_prob_nn, y_pred=y_pred_nn, uses_scaled=True
)
print(f'\n{"Neural Network (MLP)":<32}  ROC={roc_nn:.4f}  PR={pr_nn:.4f}  '
      f'F1={f1_nn:.4f}  Sen={sen_nn:.4f}  Spec={spe_nn:.4f}')

## 9. Hyperparameter Optimization — Optuna (XGBoost & LightGBM)

In [ ]:
def xgb_objective(trial):
    p = dict(
        n_estimators     = trial.suggest_int('n_estimators', 200, 800),
        learning_rate    = trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        max_depth        = trial.suggest_int('max_depth', 3, 9),
        subsample        = trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree = trial.suggest_float('colsample_bytree', 0.5, 1.0),
        min_child_weight = trial.suggest_int('min_child_weight', 1, 30),
        gamma            = trial.suggest_float('gamma', 0.0, 2.0),
        reg_alpha        = trial.suggest_float('reg_alpha', 0.0, 2.0),
        reg_lambda       = trial.suggest_float('reg_lambda', 0.5, 3.0),
    )
    m = xgb.XGBClassifier(**p, scale_pos_weight=scale_pos_weight,
                            eval_metric='logloss', random_state=42, n_jobs=-1, verbosity=0)
    m.fit(X_train_imp, y_train)
    return roc_auc_score(y_val, m.predict_proba(X_val_imp)[:, 1])


print('Optimising XGBoost (60 trials)...')
study_xgb = optuna.create_study(direction='maximize', study_name='xgb_tia')
study_xgb.optimize(xgb_objective, n_trials=60, show_progress_bar=True)
print(f'Best XGBoost val ROC-AUC: {study_xgb.best_value:.4f}')

best_xgb = xgb.XGBClassifier(
    **study_xgb.best_params,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss', random_state=42, n_jobs=-1, verbosity=0
)
fit_and_score('XGBoost (Optuna)', best_xgb, X_train_imp, y_train, X_val_imp, y_val, use_scaled=False)

In [ ]:
def lgb_objective(trial):
    p = dict(
        n_estimators     = trial.suggest_int('n_estimators', 200, 800),
        learning_rate    = trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        num_leaves       = trial.suggest_int('num_leaves', 15, 127),
        max_depth        = trial.suggest_int('max_depth', 3, 10),
        subsample        = trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree = trial.suggest_float('colsample_bytree', 0.5, 1.0),
        min_child_samples= trial.suggest_int('min_child_samples', 5, 60),
        reg_alpha        = trial.suggest_float('reg_alpha', 0.0, 2.0),
        reg_lambda       = trial.suggest_float('reg_lambda', 0.0, 2.0),
    )
    m = lgb.LGBMClassifier(**p, class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1)
    m.fit(X_train_imp, y_train,
          eval_set=[(X_val_imp, y_val)],
          callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
    return roc_auc_score(y_val, m.predict_proba(X_val_imp)[:, 1])


print('Optimising LightGBM (60 trials)...')
study_lgb = optuna.create_study(direction='maximize', study_name='lgb_tia')
study_lgb.optimize(lgb_objective, n_trials=60, show_progress_bar=True)
print(f'Best LightGBM val ROC-AUC: {study_lgb.best_value:.4f}')

best_lgb = lgb.LGBMClassifier(
    **study_lgb.best_params,
    class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1
)
fit_and_score('LightGBM (Optuna)', best_lgb, X_train_imp, y_train, X_val_imp, y_val, use_scaled=False)

## 10. Model Comparison & Validation Plots

In [ ]:
metrics_df = pd.DataFrame({
    name: {k: v for k, v in res.items() if k in ['roc_auc', 'pr_auc', 'f1', 'sensitivity', 'specificity']}
    for name, res in results.items()
}).T.sort_values('roc_auc', ascending=False)

print('=== Validation Set Performance Summary ===')
print(metrics_df.to_string(float_format=lambda x: f'{x:.4f}'))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ROC curves
ax = axes[0]
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_val, res['y_prob'])
    ax.plot(fpr, tpr, lw=1.5, label=f"{name} ({res['roc_auc']:.3f})")
ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set(xlabel='False Positive Rate', ylabel='True Positive Rate', title='ROC Curves (Validation)')
ax.legend(fontsize=7, loc='lower right')

# Precision-Recall curves
ax = axes[1]
for name, res in results.items():
    prec, rec, _ = precision_recall_curve(y_val, res['y_prob'])
    ax.plot(rec, prec, lw=1.5, label=f"{name} ({res['pr_auc']:.3f})")
ax.set(xlabel='Recall', ylabel='Precision', title='Precision-Recall Curves (Validation)')
ax.legend(fontsize=7)

# Bar chart comparison
ax = axes[2]
metrics_df[['roc_auc', 'pr_auc', 'f1']].plot(kind='bar', ax=ax, rot=45, edgecolor='black')
ax.set(title='Metric Comparison (Validation)', ylim=(0, 1))
ax.legend(['ROC-AUC', 'PR-AUC', 'F1'])

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## 11. Final Evaluation on Held-Out Test Set

The best model (by validation ROC-AUC) is evaluated once on the test set. Classification threshold is selected via Youden's J statistic on the validation ROC curve (maximises sensitivity + specificity simultaneously).

In [ ]:
best_name  = metrics_df.index[0]
best_res   = results[best_name]
best_model = best_res['model']
print(f'Best model: {best_name}  (Val ROC-AUC = {best_res["roc_auc"]:.4f})')

# Optimal threshold (Youden's J on validation ROC)
fpr_v, tpr_v, thr_v = roc_curve(y_val, best_res['y_prob'])
j_scores    = tpr_v - fpr_v
opt_thr     = float(thr_v[np.argmax(j_scores)])
print(f'Optimal threshold (Youden J): {opt_thr:.4f}')

# Test set predictions
X_test_final = X_test_imp if not best_res['uses_scaled'] else X_test_sc

if best_name == 'Neural Network (MLP)':
    best_model.eval()
    with torch.no_grad():
        logits_t = best_model(torch.FloatTensor(X_test_final).to(DEVICE)).cpu()
    y_test_prob = torch.sigmoid(logits_t).numpy()
else:
    y_test_prob = best_model.predict_proba(X_test_final)[:, 1]

y_test_pred = (y_test_prob >= opt_thr).astype(int)

test_roc = roc_auc_score(y_test, y_test_prob)
test_pr  = average_precision_score(y_test, y_test_prob)
test_f1  = f1_score(y_test, y_test_pred, zero_division=0)
test_sen = recall_score(y_test, y_test_pred, zero_division=0)
test_spe = recall_score(y_test, y_test_pred, pos_label=0, zero_division=0)

print(f'\n{"="*55}')
print(f' FINAL TEST SET RESULTS — {best_name}')
print(f'{"="*55}')
print(f'  ROC-AUC     : {test_roc:.4f}')
print(f'  PR-AUC      : {test_pr:.4f}')
print(f'  F1-Score    : {test_f1:.4f}')
print(f'  Sensitivity : {test_sen:.4f}  (recall for TIA cases)')
print(f'  Specificity : {test_spe:.4f}')
print(f'{"="*55}')
print(classification_report(y_test, y_test_pred,
                             target_names=['No Stroke/TIA', 'Stroke/TIA']))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_test_pred,
    display_labels=['No TIA', 'TIA'],
    ax=axes[0], colorbar=False
)
axes[0].set_title(f'Confusion Matrix\n{best_name}')

fpr_t, tpr_t, _ = roc_curve(y_test, y_test_prob)
axes[1].plot(fpr_t, tpr_t, 'b-', lw=2, label=f'Test ROC-AUC = {test_roc:.3f}')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].set(xlabel='FPR', ylabel='TPR', title='Test Set ROC Curve')
axes[1].legend()
plt.tight_layout()
plt.savefig('test_set_evaluation.png', dpi=120, bbox_inches='tight')
plt.show()

## 12. Model Interpretability — SHAP

SHAP (SHapley Additive exPlanations) provides feature-level explanations required for clinical credibility.

In [ ]:
print(f'Computing SHAP values for {best_name}...')
N_SHAP = min(1000, len(X_test_final))  # sample for speed
X_shap = X_test_final[:N_SHAP]

if best_name in ('Neural Network (MLP)',):
    bg = shap.sample(X_train_sc, 100)
    explainer = shap.KernelExplainer(
        lambda x: torch.sigmoid(
            best_model(torch.FloatTensor(x).to(DEVICE))
        ).detach().cpu().numpy(),
        bg
    )
    shap_vals = explainer.shap_values(X_shap[:200])  # smaller for KernelExplainer
    X_shap = X_shap[:200]
elif isinstance(best_model, LogisticRegression):
    explainer = shap.LinearExplainer(best_model, X_train_sc)
    shap_vals = explainer.shap_values(X_shap)
else:
    explainer = shap.TreeExplainer(best_model)
    shap_vals = explainer.shap_values(X_shap)
    if isinstance(shap_vals, list):
        shap_vals = shap_vals[1]  # class 1 for RF

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

plt.sca(axes[0])
shap.summary_plot(shap_vals, X_shap, feature_names=FEATURE_COLS,
                  plot_type='bar', show=False, max_display=15)
axes[0].set_title('Top 15 Features — Mean |SHAP|')

plt.sca(axes[1])
shap.summary_plot(shap_vals, X_shap, feature_names=FEATURE_COLS,
                  show=False, max_display=15)
axes[1].set_title('SHAP Beeswarm — Feature Direction & Magnitude')

plt.tight_layout()
plt.savefig('shap_summary.png', dpi=120, bbox_inches='tight')
plt.show()

## 13. Influenza Vaccine Effect — Research Hypothesis Analysis

In [ ]:
flu_idx    = FEATURE_COLS.index('flu_vaccine') if 'flu_vaccine' in FEATURE_COLS else None
senior_idx = FEATURE_COLS.index('is_senior')   if 'is_senior'   in FEATURE_COLS else None

if flu_idx is not None:
    flu_shap   = shap_vals[:, flu_idx]
    flu_values = X_shap[:, flu_idx]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # SHAP dependence for flu vaccine
    shap.dependence_plot('flu_vaccine', shap_vals, X_shap,
                          feature_names=FEATURE_COLS, ax=axes[0], show=False)
    axes[0].set_title('Flu Vaccine SHAP Dependence on TIA Risk')

    # Vaccine effect stratified by senior status
    if senior_idx is not None:
        senior_vals = X_shap[:, senior_idx]
        sc = axes[1].scatter(flu_values, flu_shap,
                              c=senior_vals, cmap='RdYlGn_r', alpha=0.4, s=12)
        plt.colorbar(sc, ax=axes[1], label='Is Senior (65+)')
        axes[1].axhline(0, color='k', linestyle='--', linewidth=0.8)
        axes[1].set(xlabel='Flu Vaccine (0=No, 1=Yes)',
                    ylabel='SHAP value → contribution to TIA risk',
                    title='Vaccine Effect by Age Group')

    # Mean SHAP by vaccine group & age
    shap_df = pd.DataFrame({
        'shap_flu': flu_shap,
        'flu_vaccine': flu_values,
        'is_senior': X_shap[:, senior_idx] if senior_idx else np.nan
    })
    grp = shap_df.groupby(['flu_vaccine', 'is_senior'])['shap_flu'].mean().unstack()
    grp.index = ['No Vaccine', 'Vaccinated']
    grp.columns = ['< 65 yrs', '≥ 65 yrs']
    grp.plot(kind='bar', ax=axes[2], rot=0, edgecolor='black')
    axes[2].axhline(0, color='k', linestyle='--', linewidth=0.8)
    axes[2].set(title='Mean SHAP: Vaccine x Age Group', ylabel='Mean SHAP (TIA risk contribution)')

    plt.suptitle('Research Hypothesis: Flu Vaccine Effect on TIA Risk', fontsize=13)
    plt.tight_layout()
    plt.savefig('flu_vaccine_effect.png', dpi=120, bbox_inches='tight')
    plt.show()

    print('Flu vaccine SHAP summary:')
    print(f'  Overall mean SHAP:       {flu_shap.mean():+.4f}')
    print(f'  Vaccinated group:        {flu_shap[flu_values == 1].mean():+.4f}')
    print(f'  Un-vaccinated group:     {flu_shap[flu_values == 0].mean():+.4f}')
    if senior_idx is not None:
        mask65 = senior_vals == 1
        print(f'  Vaccinated + Senior 65+: {flu_shap[mask65 & (flu_values == 1)].mean():+.4f}')
        print(f'  Vaccinated + Under 65:   {flu_shap[~mask65 & (flu_values == 1)].mean():+.4f}')

## 14. Cross-Validated Performance (5-Fold)

In [ ]:
from sklearn.model_selection import cross_val_score

print('5-fold stratified cross-validation on full (train+val) set...')
X_dev_imp = np.vstack([X_train_imp, X_val_imp])
y_dev     = np.concatenate([y_train, y_val])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Re-instantiate best model with same params for CV
if 'XGBoost' in best_name:
    cv_model = xgb.XGBClassifier(
        **study_xgb.best_params if 'Optuna' in best_name else {},
        scale_pos_weight=scale_pos_weight,
        eval_metric='logloss', random_state=42, n_jobs=-1, verbosity=0
    )
    X_dev_cv = X_dev_imp
elif 'LightGBM' in best_name:
    cv_model = lgb.LGBMClassifier(
        **study_lgb.best_params if 'Optuna' in best_name else {},
        class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1
    )
    X_dev_cv = X_dev_imp
else:
    cv_model = best_model
    X_dev_cv = np.vstack([X_train_sc, X_val_sc])

cv_scores = cross_val_score(cv_model, X_dev_cv, y_dev,
                             scoring='roc_auc', cv=cv, n_jobs=-1)
print(f'CV ROC-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'Fold scores: {[f"{s:.4f}" for s in cv_scores]}')

## 15. Save Model Artifacts & Production Inference

In [ ]:
artifacts = dict(
    model_name       = best_name,
    model            = best_model,
    imputer          = imputer,
    scaler           = scaler,
    feature_cols     = FEATURE_COLS,
    optimal_threshold= opt_thr,
    val_roc_auc      = best_res['roc_auc'],
    test_roc_auc     = test_roc,
    test_pr_auc      = test_pr,
    test_sensitivity = test_sen,
    test_specificity = test_spe,
    scale_pos_weight = scale_pos_weight,
    n_train          = len(y_train),
    n_features       = len(FEATURE_COLS),
)

with open('tia_prediction_model.pkl', 'wb') as f:
    pickle.dump(artifacts, f)

print('Saved: tia_prediction_model.pkl')
print(f'\n{"="*60}')
print(f' PRODUCTION MODEL CARD')
print(f'{"="*60}')
print(f'  Model         : {best_name}')
print(f'  Training N    : {len(y_train):,}')
print(f'  Features      : {len(FEATURE_COLS)}')
print(f'  Val  ROC-AUC  : {best_res["roc_auc"]:.4f}')
print(f'  Test ROC-AUC  : {test_roc:.4f}')
print(f'  Test PR-AUC   : {test_pr:.4f}')
print(f'  Test Sensitivity: {test_sen:.4f}')
print(f'  Test Specificity: {test_spe:.4f}')
print(f'  Decision thr  : {opt_thr:.4f}  (Youden J)')
print(f'{"="*60}')

In [ ]:
def predict_tia_risk(patient: dict, artifact_path: str = 'tia_prediction_model.pkl') -> dict:
    """
    Predict TIA risk for one patient.

    Parameters
    ----------
    patient : dict — keys matching FEATURE_COLS; missing values handled by imputer.

    Returns
    -------
    dict with keys: risk_score, high_risk, risk_label
    """
    with open(artifact_path, 'rb') as f:
        art = pickle.load(f)

    row  = pd.DataFrame([{feat: patient.get(feat, np.nan) for feat in art['feature_cols']}])
    rimp = art['imputer'].transform(row)

    model = art['model']
    uses_scaled = art.get('uses_scaled', True)

    if uses_scaled:
        rinput = art['scaler'].transform(rimp)
    else:
        rinput = rimp

    if isinstance(model, nn.Module):
        model.eval()
        with torch.no_grad():
            logit = model(torch.FloatTensor(rinput).to(DEVICE)).cpu()
        prob = float(torch.sigmoid(logit).item())
    else:
        prob = float(model.predict_proba(rinput)[0, 1])

    return dict(
        risk_score  = prob,
        high_risk   = prob >= art['optimal_threshold'],
        risk_label  = 'HIGH' if prob >= art['optimal_threshold'] else 'LOW',
    )


# Example inference — 72-year-old male, hypertensive, diabetic, flu-vaccinated
example = dict(
    age=72, age_sq=72**2, is_senior=1, male=1,
    flu_vaccine=1, flu_x_senior=1,
    hypertension=1, diabetes=1,
    bmi=29.0, total_chol=215,
    cardiac_score=1, in_poverty=0, low_education=0,
    ever_smoked=1, current_smoker=0,
    race_white=1
)
result = predict_tia_risk(example)
print('Example patient: 72yo male, hypertensive, diabetic, flu-vaccinated')
print(f'  Risk score : {result["risk_score"]:.3f}')
print(f'  Risk label : {result["risk_label"]}')
print(f'  High risk  : {result["high_risk"]}')

---
## Summary

| Stage | Detail |
|---|---|
| **Data** | NHANES 2015-16, 2017-18, 2017-20 pooled; stroke/TIA outcome (MCQ160F) |
| **Target** | First occurrence of stroke/TIA (binary) |
| **Key predictor** | Influenza vaccination (IMQ011) + age interaction |
| **Models** | LR · DT · RF · GBM · XGBoost · LightGBM · PyTorch MLP |
| **Tuning** | Optuna (60 trials) for XGBoost & LightGBM |
| **Imbalance** | SMOTE (train only) + `scale_pos_weight` / `class_weight='balanced'` |
| **Interpretability** | SHAP TreeExplainer / LinearExplainer |
| **Threshold** | Youden's J (maximises sensitivity + specificity) |

> **Clinical note:** False negatives (missed TIA patients) carry higher clinical cost than false positives.  
> Consider lowering the threshold further if sensitivity is the priority in deployment.